In [1]:

"""
MAYO İmza Şeması: "Whipping" Tekniğiyle Genişletilmiş UOV Simülasyonu
==========================================================================

Bu modül, MAYO (Multivariate quadratic Algorithm YOung -- literatürde
"Practical Post-Quantum Signatures from the UOV Trapdoor" makalesiyle
tanıtılan şema) imza şemasının öğretici bir SageMath simülasyonunu
içermektedir.

MAYO, Klasik UOV'den Nasıl Farklıdır?
------------------------------------------
Klasik UOV'nin temel zayıflığı, güvenli parametrelerde açık anahtarın
ÇOK BÜYÜK olmasıdır (n = v + o değişken sayısı büyüdükçe P
matrislerinin boyutu karesel olarak büyür). MAYO bu sorunu şu fikirle
çözer: KÜÇÜK bir temel UOV örneği (n = v + o değişken, m denklem)
alınır, ancak imzalama sırasında bu küçük örnek "WHIPPING" (çırpma)
adı verilen bir teknikle KATLANARAK (k kopyasının F_q^m cisim
genişlemesi üzerinden birleştirilmesiyle) GENİŞLETİLİR. Bu sayede açık
anahtar küçük kalır (yalnızca P1, P2, P3 blokları saklanır), ancak
imzalama sırasında etkin değişken sayısı N = k*n'e çıkarılarak, m
denklemlik sistemin çözülebilir olması garanti edilir. Bu, "küçük
anahtar, orta boy imza" dengesini sağlayan MAYO'nun temel motivasyonudur.

Whipping (Çırpma) Mekanizması ve E Matrisleri
---------------------------------------------------
MAYO'nun kalbi, F_q üzerindeki m-boyutlu bir cisim genişlemesi olan
F_{q^m} = F_q[z]/(f(z))'dir (f(z), m. dereceden indirgenemez bir
polinomdur). Bu genişlemenin üreteci alpha ile ÇARPMA işlemi, F_q
üzerinde m x m boyutunda bir DOĞRUSAL dönüşüm (matris) olarak temsil
edilebilir -- bu matrise E (temel çarpım matrisi) denir ve
get_matrix_of_mult_by_alpha metoduyla hesaplanır. k adet whipping
bloğu için gereken TÜM E_{i,j} (i <= j <= k) matrisleri, bu temel E
matrisinin ARDIŞIK KUVVETLERİ olarak elde edilir (E^0, E^1, E^2, ...);
bu, F_{q^m}'nin çarpımsal yapısının E matrisinin kuvvetlerine izomorf
olmasının doğrudan bir sonucudur. Genişletilmiş açık anahtar P*,
    P*(X_1, ..., X_k) = Sum_{i<=j} E_{i,j} . P'_{i,j}(X_i, X_j)
biçiminde, k(k+1)/2 adet E matrisi ile küçük UOV bileşenlerinin (P(x_i)
köşegen terimleri, P'(x_i,x_j) polar/çift-doğrusal terimleri) doğrusal
kombinasyonu olarak inşa edilir. Sınıfın __init__ metodunda denetlenen
k(k+1)/2 < m koşulu, bu E matrislerinin DOĞRUSAL BAĞIMSIZ kalmasını
(dolayısıyla whipping işleminin bilgi kaybına yol açmamasını) sağlamak
için gerekli bir güvenlik/doğruluk ön koşuludur.

Upper Dönüşümü: Kuadratik Formun Kanonik Temsili
------------------------------------------------------
Bir kuadratik form x^T M x, M matrisinin ÜST ÜÇGENSEL kısmı ile ALT
ÜÇGENSEL kısmının TOPLAMI aynı forma karşılık geldiğinden (çünkü
x_i*x_j = x_j*x_i), kanonik (tekil) bir temsile ulaşmak için alt
üçgensel girdiler üst üçgene "katlanır" ve alt üçgen sıfırlanır. Upper
metodu tam olarak bunu yapar: U[r,c] = M[r,c] + M[c,r] (r < c için) ve
U[r,r] = M[r,r] (köşegen değişmeden kalır) tanımıyla, x^T U x = x^T M x
eşitliğini koruyan üst üçgensel bir U matrisi üretir. Bu dönüşüm,
generate_keys içinde P3 bloğunun (Oil x Oil kısmının) hesaplanmasında
kritik rol oynar: Q = O^T*P1*O + O^T*P2 negatiflenip Upper ile üst
üçgensel forma sokularak, O_bar^T * P_full * O_bar'ın alterne (gizli
Oil uzayında özdeş sıfır) olması garanti edilir -- bu, Modern UOV'deki
P3 türetiminin MAYO'daki (Upper dönüşümü aracılığıyla ifade edilen)
karşılığıdır.

o < m Durumu: MAYO'ya Özgü Esneklik
----------------------------------------
Klasik UOV'de m = o (denklem sayısı, Oil değişken sayısına eşit)
olması ZORUNLUDUR (aksi halde P3 bloğunun kare olmaması veya sistemin
tutarsız kalması söz konusu olur). MAYO ise whipping tekniği sayesinde
o < m durumuna İZİN VERİR: __init__ içinde denetlenen k*o >= m koşulu,
imzalama sırasında (k adet Oil bloğunun birleştirilmesiyle) toplam
k*o adet Oil bilinmeyeninin, m denklemlik sistemi çözmeye yetecek
kadar (ALTINDA KALMAYACAK biçimde) olmasını sağlar. Bu esneklik,
MAYO'nun küçük o değerleriyle (dolayısıyla küçük P3 bloklarıyla, küçük
açık anahtarla) çalışabilmesinin matematiksel temelidir.

İmzalama: Genişletilmiş Uzayda Doğrusallaştırma
------------------------------------------------------
İmzalama, Modern UOV'deki doğrusallaştırma fikrinin k KOPYALI
(whipped) versiyonudur: k adet bağımsız rastgele Vinegar vektörü
(v_1, ..., v_k) seçilir; bunların P* üzerindeki sabit katkısı (y)
hesaplanır; ardından k*o boyutunda bilinmeyen (tüm blokların Oil
kısımları x_1, ..., x_k birleştirilmiş) içeren TEK BİR doğrusal sistem
L * x = t - y kurulur ve çözülür. L matrisinin bloklar arası yapısı,
E_{i,j} matrislerinin S_vv, S_vo alt bloklarıyla ağırlıklandırılmış
kombinasyonlarından oluşur -- bu, whipping tekniğinin imzalama
aşamasındaki doğrudan yansımasıdır. Sistem çözülürse, her blok için
imza s_i = (v_i + O*x_i) || x_i biçiminde oluşturulup birleştirilir.

Bu Simülasyonun Sınırları ve Amaçları
-----------------------------------------
- Bu kod GÜVENLİ parametreler üretmek için değil, MAYO'nun whipping
  tekniği, E matrislerinin cebirsel kökeni, Upper dönüşümü ve o < m
  esnekliğinin işleyişini somut biçimde göstermek amacıyla
  hazırlanmıştır.
- Küçük bir sonlu cisim (örn. GF(7)) ve az sayıda değişken/whipping
  faktörü üzerinde çalıştırılarak ara adımların (E matrisleri, P
  blokları, P* polinomları, doğrusal sistemler) okunabilir biçimde
  ekrana yazdırılması hedeflenmiştir.
- İmzalama fonksiyonu (sign), her denemede rastgele Vinegar vektörleri
  seçtiğinden BAŞARISIZ olabilir (L matrisi tekil çıkabilir); bu
  nedenle en fazla 100 deneme yapılan bir RETRY mekanizması içerir --
  bu, Oil-Vinegar ailesi şemalarının DOĞASINDA olan, kriptografik
  güvenliği etkilemeyen normal bir durumdur.

Algoritmanın Genel Akışı
--------------------------
1. Anahtar Üretimi (generate_keys):
   Whipping E matrisleri (temel çarpım matrisinin kuvvetleri), gizli
   O matrisi ve küçük UOV bileşenleri (P1, P2, P3 blokları) üretilir;
   bunlardan genişletilmiş açık anahtar (P_star_polys) whipping
   formülüyle inşa edilir.

2. İmzalama (sign):
   k adet rastgele Vinegar vektörü seçilir; genişletilmiş uzayda tek
   bir doğrusal sistem (L*x = t-y) kurulup çözülerek Oil ön-görüntüleri
   bulunur ve k adet UOV imza bloğu birleştirilerek nihai imza elde
   edilir.

3. Doğrulama (verify):
   İmza, genişletilmiş açık anahtar polinomlarında (P_star_polys)
   doğrudan değerlendirilip sonucun orijinal mesajla eşleşip
   eşleşmediği kontrol edilir.

Bu implementasyon; yüksek lisans tezi kapsamında MAYO'nun whipping
tabanlı sıkıştırma/genişletme mekanizmasını ve o < m esnekliğini
somutlaştırmak amacıyla eğitim/araştırma amaçlı geliştirilmiştir.


-------------
Bu kod SageMath ortamında çalıştırılmak üzere tasarlanmıştır ve sonlu
cisimler (GF), cisim genişlemeleri (FiniteField.extension), çok
değişkenli polinom halkaları (PolynomialRing), blok matris inşası
(block_matrix), matris cebiri (alt matris çıkarma, kuvvet alma,
doğrusal sistem çözümü) gibi SageMath'in yerleşik simgesel/sayısal
cebir araçlarına dayanır.

Referans: Tez, Bölüm 6 
"""

from sage.all import *

class MAYOSignatureSchemeDemo:
    """
    MAYO çok değişkenli kuadratik (MQ) imza şemasının, whipping
    (çırpma) tekniğiyle küçük bir UOV örneğini genişleten öğretici
    SageMath simülasyonu.

    Bu sınıf; F_q üzerinde küçük bir UOV örneği (P1, P2, P3 blokları)
    ve bu örneği k kat genişletmek için gereken whipping (E) matrislerini
    üretir; bunlardan genişletilmiş açık anahtar P*'ı inşa eder;
    ardından genişletilmiş uzayda doğrusallaştırma ile imzalama ve açık
    anahtar polinomlarının değerlendirilmesiyle doğrulama işlemlerini
    gerçekleştirir.

    Matematiksel Kurulum
    ---------------------
    - F_q = GF(q) : Temel sonlu cisim.
    - F_{q^m} : F_q üzerinde m. dereceden cisim genişlemesi; whipping
      E matrislerinin cebirsel kökenini oluşturur (alpha ile çarpımın
      F_q-doğrusal matris temsili).
    - v, o : Küçük (temel) UOV örneğindeki Vinegar ve Oil değişken
      sayıları; n = v + o toplam değişken sayısıdır.
    - m : Denklem sayısı (P ve P* polinomlarının sayısı); MAYO'da
      o < m olabilir (klasik UOV'nin aksine).
    - k : Whipping (genişletme) faktörü; genişletilmiş toplam değişken
      sayısı N = n*k'dir.
    - R_UOV = F_q[x_0, ..., x_{n-1}] : Küçük UOV örneğinin polinom
      halkası.
    - R_MAYO = F_q[X_0, ..., X_{N-1}] : Genişletilmiş (whipped) açık
      anahtarın tanımlı olduğu polinom halkası.

    Parametreler
    ------------
    q : int
        Temel sonlu cismin eleman sayısı (asal veya asal kuvveti).
    v : int
        Küçük UOV örneğindeki Vinegar değişkenlerinin sayısı.
    o : int
        Küçük UOV örneğindeki Oil değişkenlerinin (gizli uzay
        boyutunun) sayısı; MAYO'da o < m olabilir.
    m : int
        Denklem sayısı (açık anahtar bileşeni sayısı).
    k : int
        Whipping (genişletme) faktörü; k(k+1)/2 adet E matrisi
        gerektirir ve k(k+1)/2 < m koşulunun sağlanması önerilir.

    Öznitelikler (Attributes)
    --------------------------
    n : int
        Küçük UOV örneğindeki toplam değişken sayısı (v + o).
    total_n : int
        Genişletilmiş (whipped) toplam değişken sayısı (n * k),
        kısaca N olarak da anılır.
    F : FiniteField
        Temel sonlu cisim F_q.
    F_qm : FiniteField
        F_q üzerinde m. dereceden cisim genişlemesi F_{q^m}.
    alpha : FiniteFieldElement
        F_qm cisminin üreteci; whipping E matrislerinin türetildiği
        çarpım işleminin tabanı.
    R_UOV : MPolynomialRing
        n değişkenli, F_q katsayılı, küçük UOV örneğinin polinom halkası.
    vars_UOV : vector
        R_UOV halkasının üreteçlerinden oluşan vektör.
    R_MAYO : MPolynomialRing
        total_n değişkenli, F_q katsayılı, genişletilmiş açık anahtarın
        polinom halkası.
    vars_MAYO : vector
        R_MAYO halkasının üreteçlerinden oluşan vektör.
    O_mat : Matrix
        Gizli anahtarı oluşturan v x o boyutunda matris (küçük UOV
        örneğinin gizli Oil uzayını tanımlar).
    P_matrices : list of Matrix
        Küçük UOV örneğinin açık anahtar matrisleri (her biri n x n
        boyutunda, m adet).
    E_matrices : dict
        (i, j) çiftlerini (0 <= i <= j < k) F_q üzerinde m x m boyutunda
        whipping matrislerine eşleyen sözlük.
    P_star_polys : list of MPolynomial
        Genişletilmiş açık anahtarı oluşturan, R_MAYO halkasına ait
        m adet polinom.
    """
    def __init__(self, q, v, o, m, k):
        """
        MAYO - Tez İçin Detaylı Simülasyon
        Özellikler:
        1. Cebirsel E Matrisi (F_q^m genişlemesi üzerinden)
        2. Upper Dönüşümü ile P3 Hesabı
        3. o < m durumu destekli (MAYO özelliği)
        4. Her adımda detaylı çıktı

        MAYO şemasının parametrelerini, temel sonlu cismi, whipping
        için gerekli cisim genişlemesini ve iki ayrı polinom halkasını
        (küçük UOV örneği için R_UOV, genişletilmiş açık anahtar için
        R_MAYO) başlatır. Ayrıca MAYO'nun doğru çalışması için gerekli
        iki temel parametre koşulunu (k(k+1)/2 < m ve k*o >= m)
        denetleyip sonucu ekrana bilgilendirme amaçlı yazdırır.

        Parametreler
        ------------
        q : int
            Temel sonlu cismin eleman sayısı.
        v : int
            Küçük UOV örneğindeki Vinegar değişkenlerinin sayısı.
        o : int
            Küçük UOV örneğindeki Oil değişkenlerinin sayısı.
        m : int
            Denklem sayısı.
        k : int
            Whipping (genişletme) faktörü.

        Yan Etkiler
        -----------
        self.F, self.F_qm, self.alpha, self.R_UOV, self.vars_UOV,
        self.R_MAYO, self.vars_MAYO öznitelikleri hesaplanır; self.n =
        v + o, self.total_n = n * k olarak belirlenir; self.O_mat None,
        self.P_matrices/self.E_matrices/self.P_star_polys boş
        koleksiyonlar olarak başlatılır (bunlar ancak generate_keys()
        çağrıldığında doldurulur). Parametre özeti ve ön koşul
        denetimleri ekrana yazdırılır.

        Dönüş Değeri
        ------------
        None
        """
        self.q = q
        self.v = v
        self.o = o
        self.m = m
        self.k = k
        self.n = v + o

        print("\n" + "="*80)
        print(f"MAYO İMZA ŞEMASI: TEZ SİMÜLASYONU ")
        print(f"Parametreler: GF({q}), v={v}, o={o}, m={m}")
        print(f"Whipping (k): {k} => Genişletilmiş Uzay Boyutu N = {self.n * k}")

        # MAYO Şartı Kontrolü
        # k(k+1)/2, whipping için gerekli E_{i,j} (i<=j) matrislerinin
        # toplam sayısıdır. Bu sayının m'den KÜÇÜK kalması, E
        # matrislerinin F_{q^m} cisminin çarpımsal yapısı içinde
        # DOĞRUSAL BAĞIMSIZ kalmasını (dolayısıyla whipping'in bilgi
        # kaybetmeden çalışmasını) sağlamak için gerekli bir koşuldur.
        num_E_matrices = (k * (k + 1)) // 2
        print(f"Gerekli E Matrisi Sayısı (k(k+1)/2): {num_E_matrices}")
        if num_E_matrices >= m:
            print(f"[UYARI] k(k+1)/2 >= m ({num_E_matrices} >= {m}). Lineer bağımsızlık riski!")
        else:
            print(f"[ONAY] k(k+1)/2 < m ({num_E_matrices} < {m}). Şart sağlanıyor.")
        print("="*80)

        # k*o, imzalama sırasında genişletilmiş uzayda ortaya çıkan
        # TOPLAM Oil bilinmeyeni sayısıdır (k blok, her biri o adet
        # Oil değişkeni). Bu sayının m denklem sayısından KÜÇÜK
        # OLMAMASI, doğrusal sistemin (L*x=t-y) altında kalınmayacak
        # (under-determined olmayacak) kadar bilinmeyen içermesini
        # garanti eder -- MAYO'nun o < m durumuna izin vermesinin
        # arkasındaki denge budur.
        if k * o < m:
            print(f"[UYARI] k*o < m ({k*o} < {m}). İmzalama sistemi aşırı tanımlı kalabilir.")
        else:
            print(f"[ONAY] k*o >= m ({k*o} >= {m}). İmzalama için yeterli bilinmeyen sayısı mevcut.")

        # 1. TEMEL CİSİM (F_q)
        self.F = GF(q, 'u')

        # 2. GENİŞLEME CİSMİ (F_qm)
        # F_q üzerine m. dereceden genişleme (E matrisleri için)
        # F_{q^m}, whipping E matrislerinin cebirsel kaynağıdır: bu
        # cismin üreteci alpha ile çarpma işlemi, F_q üzerinde m x m
        # boyutunda bir doğrusal dönüşüme (matrise) karşılık gelir ve
        # bu matrisin kuvvetleri gerekli tüm E_{i,j} matrislerini verir.
        try:
            self.F_qm = self.F.extension(m, 'a')
        except ValueError:
            # Bazen q ve m kombinasyonuna göre 'a' ismi çakışabilir, otomatik bırakıyoruz
            self.F_qm = self.F.extension(m)

        self.alpha = self.F_qm.gen()
        print(f"\n[SİSTEM] Cisim Genişlemesi F_{q}^{m} oluşturuldu.")
        print(f"         İndirgenemez Polinom: {self.F_qm.modulus()}")

        # Polinom Halkaları
        # R_UOV: küçük (whipping öncesi) UOV örneğinin tanımlı olduğu
        # n değişkenli halka; yalnızca P_matrices'in polinom karşılığı
        # gerektiğinde referans olarak kullanılabilir.
        self.R_UOV = PolynomialRing(self.F, self.n, 'x')
        self.vars_UOV = vector(self.R_UOV, self.R_UOV.gens())

        # R_MAYO: whipping sonrası GENİŞLETİLMİŞ açık anahtarın (P*)
        # tanımlı olduğu, total_n = n*k değişkenli halka; MAYO'nun
        # nihai açık anahtar polinomları bu halka üzerinde yaşar.
        self.total_n = self.n * self.k
        self.R_MAYO = PolynomialRing(self.F, self.total_n, 'X')
        self.vars_MAYO = vector(self.R_MAYO, self.R_MAYO.gens())

        self.O_mat = None
        self.P_matrices = []
        self.E_matrices = {}
        self.P_star_polys = []

    def get_matrix_of_mult_by_alpha(self):
        """
        alpha ile çarpma işleminin matris karşılığını bulur. (alpha cisim genişlemesinin üreteci olmaktadır.)

        Teorik Gerekçe
        ---------------
        F_{q^m} = F_q[z]/(f(z)) cismi, F_q üzerinde m boyutlu bir
        vektör uzayı olarak görülebilir (baz: 1, alpha, alpha^2, ...,
        alpha^{m-1}). Bu vektör uzayı üzerinde "alpha ile çarpma"
        işlemi F_q-DOĞRUSAL bir dönüşümdür (çünkü
        alpha*(a+b) = alpha*a + alpha*b ve alpha*(c*a) = c*(alpha*a)
        her c in F_q için sağlanır); dolayısıyla bu dönüşüm m x m
        boyutunda tek bir matrisle (E) tam olarak temsil edilebilir.
        Bu metot, her baz elemanı alpha^j için alpha*(alpha^j) =
        alpha^{j+1} sonucunu hesaplayıp bunun F_q üzerindeki koordinat
        vektörünü E matrisinin j'inci sütunu olarak kaydeder. Elde
        edilen E matrisinin KUVVETLERİ (E^0, E^1, E^2, ...), sırasıyla
        1, alpha, alpha^2, ... ile çarpma işlemlerine karşılık gelir
        ve generate_keys içinde tüm E_{i,j} whipping matrislerinin
        türetilmesinde temel yapı taşı olarak kullanılır.

        Dönüş Değeri
        ------------
        Matrix
            F_q üzerinde tanımlı, m x m boyutunda, alpha ile çarpma
            işlemini temsil eden doğrusal dönüşüm matrisi.
        """
        cols = []
        for j in range(self.m):
            basis_elem = self.alpha**j
            result = self.alpha * basis_elem

            poly = result.polynomial()
            coeffs = poly.list()

            # Tam m boyutunda vektör oluştur (Padding)
            vec = vector(self.F, self.m)
            for idx, val in enumerate(coeffs):
                if idx < self.m:
                    vec[idx] = self.F(val)
            cols.append(vec)

        E = matrix(self.F, cols).transpose()
        return E

    def Upper(self, M):
        """
        Üst üçgensel kuadratik temsil dönüşümü.

        Bir M matrisi için x^T M x kuadratik formunu değiştirmeden
        alt üçgensel girdileri simetrik üst üçgensel konumlara ekler
        ve alt üçgeni sıfırlar.

        Sonuçta elde edilen U üst üçgensel matris için:
            x^T U x = x^T M x
        olur.

        Teorik Gerekçe
        ---------------
        Bir kuadratik formun matris temsili TEKİL DEĞİLDİR: herhangi
        bir M matrisi için x^T M x değeri, yalnızca M'nin SİMETRİK
        kısmına (M + M^T)/2'ye, ya da eşdeğer biçimde M'nin üst
        üçgensel kısmına M[i,j]+M[j,i] (i<j) katsayılarıyla "katlanmış"
        haline bağlıdır -- çünkü x_i*x_j ve x_j*x_i monomiyeli özdeştir.
        Bu metot, M'nin ALT üçgensel girdilerini (M[c,r], r<c) karşılık
        gelen ÜST üçgensel girdiye (U[r,c] = M[r,c] + M[c,r]) toplayarak
        ve alt üçgeni sıfırlayarak KANONİK bir üst üçgensel temsile
        ulaşır. Bu dönüşüm, generate_keys içinde P3 bloğunun (Oil x Oil
        kısmının) inşasında -(O^T*P1*O + O^T*P2) ifadesinin kanonik üst
        üçgensel forma sokulması için kullanılır; bu sayede P3 hem
        tutarlı bir kuadratik form matrisi olur hem de gizli Oil
        uzayının sıfırlanma koşulunu (O_bar^T P_full O_bar'ın alterne
        olması) sağlar.

        Parametreler
        ------------
        M : Matrix
            self.F üzerinde tanımlı, dönüştürülecek kare matris
            (tipik olarak -(O^T*P1*O + O^T*P2) ara hesabı).

        Dönüş Değeri
        ------------
        Matrix
            M ile aynı boyutlarda, üst üçgensel (alt üçgeni sıfır)
            bir matris U; x^T U x = x^T M x eşitliğini sağlar.
        """
        rows, cols = M.nrows(), M.ncols()
        U = matrix(self.F, rows, cols)

        for r in range(rows):
            for c in range(r, cols):
                if r == c:
                    U[r, c] = M[r, c]
                else:
                    U[r, c] = M[r, c] + M[c, r]

        return U

    def generate_keys(self):
        """
        MAYO şemasının gizli anahtarını (whipping E matrisleri, gizli
        O matrisi, küçük UOV bileşenleri P1/P2/P3) rastgele üretir ve
        bunlardan whipping formülüyle genişletilmiş açık anahtarı
        (P_star_polys) inşa eder.

        Algoritmanın Adımları
        -----------------------
        1. Whipping (E) Matrislerinin Hesaplanması:
           get_matrix_of_mult_by_alpha ile temel çarpım matrisi
           Mat_E hesaplanır. Her (i, j) çifti (0 <= i <= j < k) için,
           bu matrisin ARDIŞIK bir kuvveti E_{i,j} = Mat_E^power olarak
           atanır (power_counter, döngü boyunca 0'dan k(k+1)/2 - 1'e
           kadar artar). Bu, F_{q^m} cisminin çarpımsal yapısındaki
           farklı kuvvetlerin (1, alpha, alpha^2, ...) F_q-doğrusal
           matris karşılıklarını verir.

        2. Gizli O Matrisi:
           v x o boyutunda rastgele bir O_mat matrisi seçilir; bu,
           küçük UOV örneğinin gizli Oil uzayını tanımlar.

        3. Temel UOV Dönüşümü (Her idx = 0, ..., m-1 için):
           P1 (v x v, üst üçgensel) ve P2 (v x o) blokları rastgele
           seçilir; P3 bloğu Upper(-(O^T*P1*O + O^T*P2)) formülüyle
           DETERMİNİSTİK olarak hesaplanır (Modern UOV'deki P3
           türetiminin Upper dönüşümü aracılığıyla ifade edilen
           eşdeğeridir); bloklar P_full = [[P1,P2],[0,P3]] biçiminde
           birleştirilerek küçük UOV açık anahtar matrisi elde edilir.

        4. P* (Genişletilmiş Açık Anahtar) Oluşturulması:
           Genişletilmiş değişken vektörü k adet n-boyutlu alt vektöre
           (X_vectors[i]) bölünür. Formül
               P*(X_1,...,X_k) = Sum_{i<=j} E_{i,j} . P'_{i,j}(X_i,X_j)
           uygulanır: i == j için köşegen terim X_i^T*M*X_i (M = küçük
           UOV matrisi), i < j için POLAR (çift-doğrusal) terim
           X_i^T*(M+M^T)*X_j hesaplanır; bu terimler m-boyutlu bir
           vektörde toplanıp E_{i,j} matrisiyle çarpılarak P* vektörüne
           eklenir. Sonuç, R_MAYO halkasına ait m adet polinom
           (P_star_polys) olarak kaydedilir.

        Yan Etkiler
        -----------
        self.E_matrices, self.O_mat, self.P_matrices ve
        self.P_star_polys öznitelikleri bu metot tarafından (yeniden)
        doldurulur. Metot birden fazla kez çağrılırsa önceki anahtar
        bileşenleri temizlenip yeniden üretilir. Süreç boyunca
        ayrıntılı ara çıktılar (E matrisleri, P blokları, ara hesap Q,
        P* polinomları) ekrana yazdırılır.

        Dönüş Değeri
        ------------
        None
        """
        # generate_keys() birden fazla kez çağrılırsa eski anahtar bileşenleri temizlenir.
        self.O_mat = None
        self.P_matrices = []
        self.E_matrices = {}
        self.P_star_polys = []

        print("\n" + "*"*40 + " BÖLÜM 1: ANAHTAR ÜRETİMİ " + "*"*40)

        # --- 1. E MATRİSLERİ ---
        print(f"\n[1.1] Whipping (E) Matrislerinin Hesaplanması")
        print(f"      Yöntem: F_q[z]/f(z) üzerindeki çarpım matrisinin kuvvetleri")

        Mat_E = self.get_matrix_of_mult_by_alpha()
        print(f"      Temel E Matrisi (alpha ile çarpımın F_q-lineer matris temsili):\n{Mat_E}")

        # Her E_{i,j} matrisi, temel çarpım matrisinin ARDIŞIK bir
        # kuvvetidir; power_counter döngü boyunca artırılarak (i,j)
        # çiftleri ile F_{q^m}'deki farklı alpha kuvvetleri arasında
        # bire bir bir eşleme kurulur.
        power_counter = 0
        for i in range(self.k):
            for j in range(i, self.k):
                E_pow = Mat_E**power_counter
                self.E_matrices[(i, j)] = E_pow

                # TÜM E MATRİSLERİNİ YAZDIR
                print(f"\n      -> E_{i+1},{j+1} = E^{power_counter}:\n{E_pow}")
                power_counter += 1
        print(f"      Toplam {len(self.E_matrices)} adet E matrisi hafızaya alındı.")

        # --- 2. GİZLİ O MATRİSİ ---
        print(f"\n[1.2] Gizli Oil Uzayı (O Matrisi)")
        self.O_mat = random_matrix(self.F, self.v, self.o)
        print(f"      Boyut: {self.v}x{self.o}")
        print(f"{self.O_mat}")

        # --- 3. TEMEL UOV (P1, P2, P3) ---
        print(f"\n[1.3] Temel UOV Dönüşümü P = (p_1, ..., p_m)")
        print(f"      Kural: P3 = Upper( - (O^T * P1 * O + O^T * P2) )")

        for idx in range(self.m):
            # P1: Vinegar x Vinegar terimleri, üst üçgensel ve tamamen
            # serbestçe rastgele seçilir.
            P1 = matrix(self.F, self.v, self.v)
            for i in range(self.v):
                for j in range(i, self.v): P1[i, j] = self.F.random_element()

            # P2: Vinegar x Oil terimleri, tamamen serbestçe rastgele seçilir.
            P2 = random_matrix(self.F, self.v, self.o)

            # P3 Hesabı
            # Q, P1 ve P2'nin gizli Oil uzayı üzerindeki kuadratik
            # forma yaptığı katkıyı temsil eder; P3, -Q'nun Upper
            # dönüşümüyle kanonik üst üçgensel forma sokulmasıyla
            # DETERMİNİSTİK olarak elde edilir; bu, O_bar^T*P_full*O_bar
            # matrisinin alterne (gizli Oil uzayında özdeş sıfır)
            # olmasını garanti eder.
            Q = self.O_mat.transpose() * P1 * self.O_mat + self.O_mat.transpose() * P2
            P3 = self.Upper(-Q)

            # Tam P Matrisi: alt sol blok özdeş sıfırdır (üst
            # üçgensel kuadratik form temsilinin doğal bir sonucu).
            Zero_blk = matrix(self.F, self.o, self.v)
            P_full = block_matrix([[P1, P2], [Zero_blk, P3]])
            self.P_matrices.append(P_full)

            # HER BİR POLİNOM İÇİN DETAYLI MATRİSLERİ YAZDIR
            print(f"\n      >>> Temel Bileşen p_{idx+1}(x) Matrisleri <<<")
            print(f"      P1_{idx+1} ({self.v}x{self.v}):\n{P1}")
            print(f"      P2_{idx+1} ({self.v}x{self.o}):\n{P2}")
            print(f"      Ara Hesap Q (O^T*P1*O + O^T*P2):\n{Q}")
            print(f"      P3_{idx+1} = Upper(-Q) ({self.o}x{self.o}):\n{P3}")
            print(f"      Tam Blok Matris P_{idx+1}:\n{P_full}")

        # --- 4. P* HESABI ---
        print(f"\n[1.4] P* (Genişletilmiş Açık Anahtar) Oluşturuluyor...")
        print(f"      Formül: P*(x_1...x_k) = Σ E_ii P(x_i) + Σ E_ij P'(x_i, x_j)")

        # Genişletilmiş değişken vektörü, her biri n boyutunda k adet
        # alt vektöre (bloğa) bölünür; her blok, whipping işleminin
        # bağımsız bir "kopyasını" temsil eder.
        X_vectors = []
        for i in range(self.k):
            start = i * self.n
            X_vectors.append(self.vars_MAYO[start : start + self.n])

        P_star_vector = vector(self.R_MAYO, self.m)
        for i in range(self.k):
            for j in range(i, self.k):
                E = self.E_matrices[(i, j)]
                temp_P_vec = vector(self.R_MAYO, self.m)
                for eq_idx in range(self.m):
                    M = self.P_matrices[eq_idx]
                    # i == j için P(x_i) hesaplanır.
                    # i < j için polar form P'(x_i,x_j)=x_i^T(P+P^T)x_j hesaplanır.
                    if i == j: val = X_vectors[i] * M * X_vectors[i]
                    else:      val = X_vectors[i] * (M + M.transpose()) * X_vectors[j]
                    temp_P_vec[eq_idx] = val
                P_star_vector += E * temp_P_vec
        self.P_star_polys = list(P_star_vector)

        print(f"      -> {len(self.P_star_polys)} adet P* polinomu oluşturuldu.")

        # TÜM P* POLİNOMLARINI YAZDIR
        for i, poly in enumerate(self.P_star_polys):
            print(f"\n      P*_{i+1}(X) Polinomu:\n{poly}")



    def _check_keys_generated(self):
        """
        MAYO açık anahtarının ve gizli O matrisinin üretilip üretilmediğini kontrol eder.

        Bu iç yardımcı metot, sign() ve verify() metotlarının
        generate_keys() çağrılmadan kullanılmasını engelleyen bir
        koruma (guard) katmanıdır; O_mat, P_matrices, E_matrices veya
        P_star_polys bileşenlerinden herhangi biri eksikse anlamlı bir
        hata mesajıyla erken durdurma sağlar.

        Ön Koşullar
        ------------
        Yoktur (bu metot herhangi bir zamanda çağrılabilir; amacı tam
        olarak diğer metotların ön koşulunu denetlemektir).

        Dönüş Değeri
        ------------
        None
            Anahtarlar eksiksizse sessizce döner.

        Fırlatılan İstisnalar
        -----------------------
        RuntimeError
            self.O_mat None ise, ya da self.P_matrices, self.E_matrices
            veya self.P_star_polys boşsa fırlatılır.
        """
        if self.O_mat is None or len(self.P_matrices) == 0 or len(self.E_matrices) == 0 or len(self.P_star_polys) == 0:
            raise RuntimeError("Önce generate_keys() çağrılmalıdır.")



    def sign(self, message_vec):
        """
        Verilen bir mesaj (özet) vektörü için, whipping tekniğiyle
        genişletilmiş uzayda tek bir doğrusal sistem kurup çözerek
        MAYO imzasını üretir.

        Algoritmanın Adımları
        -----------------------
        1. Vinegar Seçimi:
           k adet bağımsız rastgele Vinegar vektörü (V_vectors[0..k-1],
           her biri v boyutunda) seçilir; bunlar whipping'in k
           bağımsız "kopyasına" karşılık gelen serbest parametrelerdir.

        2. Mevcut Durumun (y) Hesaplanması:
           Her blok için Vinegar değerleri yerleştirilip Oil
           koordinatları sıfırlanarak sub_dict oluşturulur; bu, tüm
           P_star_polys polinomlarında yerine konarak
           y = P*(v_1,...,v_k || 0,...,0) sabit terimi hesaplanır.
           rhs = message_vec - y, çözülecek doğrusal sistemin sağ
           tarafını oluşturur.

        3. Doğrusal Sistem Matrisi L'nin Kurulması:
           L, m x (k*o) boyutunda, k adet o-sütunluk BLOKTAN oluşur.
           Her blok L_r (r = 0,...,k-1), r'inci Oil bloğunun (x_r)
           katsayılarını taşır ve TÜM i = 0,...,k-1 whipping
           indisleri üzerinden toplanarak kurulur: uygun E_{i,r}
           (veya E_{r,i}) matrisi ile, S = P_matrices[poly_idx] +
           P_matrices[poly_idx]^T simetrikleştirilmiş matrisinin
           S_vv (Vinegar x Vinegar) ve S_vo (Vinegar x Oil) alt
           blokları kullanılarak v_vec * (S_vv*O_mat + S_vo) terimi
           hesaplanır; bu, P*(v||x) ifadesinin x_r'ye göre TÜREVİNİN
           whipping ağırlıklı toplamına karşılık gelir.

        4. Doğrusal Sistemin Çözümü:
           L * x = rhs denklemi Sage'in solve_right metoduyla
           çözülür. Çözüm bulunursa oil_sol_flat (uzunluğu k*o olan,
           tüm blokların Oil değerlerini birleşik biçimde içeren
           vektör) elde edilir; L tekil ise (çözüm yoksa), yeni bir
           rastgele Vinegar seçimiyle yeniden denenir (RETRY
           mekanizması; bu, Oil-Vinegar ailesi şemalarında beklenen,
           güvenliği etkilemeyen bir durumdur).

        5. İmza Vektörünün Oluşturulması:
           Her blok i için, oil_sol_flat'ten x_vec_i (o boyutunda)
           ayrıştırılır; s_i = (v_i + O_mat*x_i) || x_i biçiminde
           birleştirilir; tüm blokların s_i'leri sırayla birleştirilip
           uzunluğu total_n = k*n olan nihai imza elde edilir.

        Ön Koşullar
        ------------
        generate_keys() metodunun bu metottan önce çağrılmış olması
        gerekir (_check_keys_generated aracılığıyla denetlenir).
        message_vec'in uzunluğu self.m'ye eşit olmalıdır.

        Parametreler
        ------------
        message_vec : vector
            Uzunluğu self.m olan, F_q üzerinde tanımlı, imzalanacak
            (tipik olarak bir hash fonksiyonundan gelen) özet vektörü.

        Dönüş Değeri
        ------------
        vector veya None
            Başarılı olursa, uzunluğu self.total_n (= k*n) olan geçerli
            bir MAYO imzası; en fazla 100 denemede hiçbir Vinegar
            seçimi L'yi tersinir/çözülebilir kılmazsa None.

        Fırlatılan İstisnalar
        -----------------------
        RuntimeError
            Anahtarlar henüz üretilmemişse (_check_keys_generated
            aracılığıyla).
        ValueError
            message_vec'in uzunluğu self.m'ye eşit değilse.
        """
        self._check_keys_generated()

        if len(message_vec) != self.m:
            raise ValueError(f"Mesaj vektörünün uzunluğu m={self.m} olmalıdır.")

        print("\n" + "*"*40 + " BÖLÜM 2: İMZALAMA SÜRECİ " + "*"*40)
        print(f"   İmzalanacak Özet Değer (t): {message_vec}")

        attempts = 0
        while attempts < 100:
            attempts += 1
            print(f"\n   --- Deneme {attempts} ---")

            # 1. VINEGAR SEÇİMİ
            # Her whipping bloğu için bağımsız bir rastgele Vinegar
            # vektörü seçilir; bu vektörler L matrisinin ve y sabit
            # teriminin belirleyicisidir.
            V_vectors = []
            for i in range(self.k):
                V_vectors.append(random_vector(self.F, self.v))
            print(f"   [2.1] Rastgele Vinegar Vektörleri (v_1...v_{self.k}):")
            for i, v in enumerate(V_vectors):
                print(f"         v_{i+1}: {v}")

            # 2. MEVCUT DURUM (y)
            # Tüm blokların Vinegar değerleri yerleştirilip Oil
            # koordinatları sıfırlanarak P*'ın x=0 noktasındaki (yani
            # yalnızca Vinegar katkısından gelen) sabit değeri y
            # hesaplanır.
            sub_dict = {}
            for i in range(self.k):
                for j in range(self.v): sub_dict[self.vars_MAYO[i*self.n + j]] = V_vectors[i][j]
                for j in range(self.v, self.n): sub_dict[self.vars_MAYO[i*self.n + j]] = 0

            y_vec = vector(self.F, [p.subs(sub_dict) for p in self.P_star_polys])
            rhs = message_vec - y_vec
            print(f"   [2.2] Mevcut P*(v) Değeri y: {y_vec}")
            print(f"         Hedef Fark (RHS = t - y): {rhs}")

            # 3. L MATRİSİ KURULUMU
            print(f"   [2.3] Lineer Sistem Matrisi L (Boyut: {self.m} x {self.k*self.o}) Kuruluyor...")

            L_blocks = []
            for r in range(self.k):
                L_r = matrix(self.F, self.m, self.o)
                for i in range(self.k):
                    # E_{i,r} matrisi yalnızca (i,r) çifti i<=r
                    # sırasında saklandığından, i>r durumunda simetrik
                    # karşılığı E_{r,i} kullanılır (E_{i,j} = E_{j,i}
                    # rolündedir çünkü polar form simetriktir).
                    if i <= r: E = self.E_matrices[(i, r)]
                    else:      E = self.E_matrices[(r, i)]

                    v_vec = V_vectors[i]
                    M_i_rows = []
                    for poly_idx in range(self.m):
                        M = self.P_matrices[poly_idx]
                        S = M + M.transpose()

                        S_vv = S.submatrix(0, 0, self.v, self.v)
                        S_vo = S.submatrix(0, self.v, self.v, self.o)

                        # Bu terim, P*(v||x) ifadesinin x_r'ye göre
                        # kısmi türevinin, v_i vektörüyle çarpılan
                        # kısmına karşılık gelir; whipping E matrisiyle
                        # ağırlıklandırılıp tüm i üzerinden toplanarak
                        # L_r bloğu inşa edilir.
                        term = v_vec * (S_vv * self.O_mat + S_vo)
                        M_i_rows.append(term)

                    M_i = matrix(self.F, M_i_rows)
                    L_r += E * M_i
                L_blocks.append(L_r)

            L = block_matrix([L_blocks], subdivide=False)
            print(f"         Oluşturulan L Matrisi:\n{L}")
            print("         Açık Lineer Denklem Sistemi:")

            unknown_names = []
            for block_idx in range(self.k):
                for oil_idx in range(self.o):
                    unknown_names.append(f"x{block_idx+1}_{oil_idx}")

            for row_idx in range(L.nrows()):
                terms = []
                for col_idx in range(L.ncols()):
                    coeff = L[row_idx, col_idx]
                    if coeff != 0:
                        terms.append(f"({coeff})*{unknown_names[col_idx]}")
                lhs = " + ".join(terms) if terms else "0"
                print(f"            Denklem {row_idx+1}: {lhs} = {rhs[row_idx]}")

            # 4. ÇÖZÜM
            try:
                oil_sol_flat = L.solve_right(rhs)
                print(f"   [2.4] ÇÖZÜM BAŞARILI! L * x = RHS sağlandı.")
                print(f"         Bulunan x (Oil Değerleri): {oil_sol_flat}")

                # 5. İMZA BİRLEŞTİRME
                print(f"   [2.5] İmza Oluşturuluyor: s_i = (v_i + O*x_i) || x_i")
                final_sig_parts = []
                for i in range(self.k):
                    x_vec_i = oil_sol_flat[i*self.o : (i+1)*self.o]
                    v_vec_i = V_vectors[i]

                    # Calculate parts
                    # Üst kısım, çözülen Oil bileşeninin O_mat
                    # aracılığıyla Vinegar koordinatlarına "geri
                    # yansıtılmış" katkısıyla birleştirilmiş halidir;
                    # bu, her bloğun gizli Oil uzayının O_bar
                    # parametrizasyonuna uyduğunu garanti eder.
                    Ox = self.O_mat * x_vec_i
                    top_part = v_vec_i + Ox
                    bottom_part = x_vec_i

                    final_sig_parts.extend(list(top_part) + list(bottom_part))

                    print(f"         Blok {i+1}:")
                    print(f"           x_{i+1}: {x_vec_i}")
                    print(f"           O*x_{i+1}: {Ox}")
                    print(f"           Üst Kısım (v_{i+1} + O*x_{i+1}): {top_part}")

                signature = vector(self.F, final_sig_parts)
                print(f"\n   [SONUÇ] İMZA (s): {signature}")
                return signature

            except ValueError:
                # L matrisi tekil/çözümsüz ise bu Vinegar seçimi işe
                # yaramaz; bu durumda tüm süreç yeni bir rastgele
                # Vinegar seçimiyle baştan denenmelidir.
                print("   [HATA] Sistem tekil veya çözüm yok. Tekrar deneniyor...")
                continue
        return None


    def verify(self, message, signature):
        """
        Verilen bir imzanın, genişletilmiş açık anahtar polinomları
        (P_star_polys) aracılığıyla verilen mesaja karşılık gelip
        gelmediğini doğrular.

        Teorik Gerekçe
        ---------------
        MAYO (ve genel olarak çok değişkenli kuadratik imza şemaları)
        tek yönlü bir kapı fonksiyonuna dayanır: P*(X)'i (gizli anahtar
        bilgisi olmadan) HERHANGİ bir hedef için tersine çevirmek
        hesaplama açısından zordur, ancak P*(X)'i belirli bir X
        noktasında DEĞERLENDİRMEK (ileri yön) kolaydır. Doğrulama, tam
        olarak bu kolay yönü kullanır: imza doğrudan tüm P_star_polys
        polinomlarında yerine konur ve sonucun mesajla eşleşip
        eşleşmediği kontrol edilir; bu adım hiçbir gizli anahtar
        bilgisi (O matrisi, E matrisleri, P1/P2/P3 blokları)
        gerektirmez -- yalnızca açık anahtar polinomları yeterlidir.

        Ön Koşullar
        ------------
        generate_keys() metodunun bu metottan önce çağrılmış olması
        gerekir (_check_keys_generated aracılığıyla denetlenir).
        message'ın uzunluğu self.m'ye, signature'ın uzunluğu
        self.total_n'ye eşit olmalıdır.

        Parametreler
        ------------
        message : vector
            Uzunluğu self.m olan, F_q üzerinde tanımlı orijinal mesaj
            (özet) vektörü.
        signature : vector
            Uzunluğu self.total_n (= k*n) olan, F_q üzerinde tanımlı,
            doğrulanacak imza vektörü (tipik olarak sign() metodunun
            çıktısı).

        Dönüş Değeri
        ------------
        bool
            İmza, P_star_polys polinomlarında değerlendirildiğinde
            mesajla tam olarak eşleşiyorsa True; aksi halde False.

        Fırlatılan İstisnalar
        -----------------------
        RuntimeError
            Anahtarlar henüz üretilmemişse (_check_keys_generated
            aracılığıyla).
        ValueError
            message'ın uzunluğu self.m'ye veya signature'ın uzunluğu
            self.total_n'ye eşit değilse.
        """
        self._check_keys_generated()

        if len(message) != self.m:
            raise ValueError(f"Mesaj vektörünün uzunluğu m={self.m} olmalıdır.")

        if len(signature) != self.total_n:
            raise ValueError(f"İmza vektörünün uzunluğu N=k*n={self.total_n} olmalıdır.")

        print("\n" + "*"*40 + " BÖLÜM 3: DOĞRULAMA " + "*"*40)
        print(f"   Gelen Mesaj (t): {message}")
        print(f"   Gelen İmza (s) : {signature}")

        val_dict = {self.vars_MAYO[i]: signature[i] for i in range(self.total_n)}
        check = vector(self.F, [p.subs(val_dict) for p in self.P_star_polys])

        print(f"   P*(s) Sonucu   : {check}")

        if check == message:
            print("\n   >>> DOĞRULAMA BAŞARILI (Geçerli İmza) <<<")
            return True
        else:
            print("\n   >>> DOĞRULAMA BAŞARISIZ (Geçersiz İmza) <<<")
            return False



In [3]:
# =====================================================================================================
# TEZ İÇİN ÖRNEK SENARYO (Bu hücrede verilen parametrelerde anahtar üretimi süreci simüle edilmektedir.
# =====================================================================================================

# Küçük öğretici parametreler:
# q = 7  : Sonlu cisim GF(7)
# v = 2  : Temel UOV dönüşümündeki vinegar değişkeni sayısı
# o = 2  : Gizli oil uzayının boyutu
# m = 4  : Denklem sayısı
# k = 2  : Whipping/genişletme faktörü
#
# Bu seçimde:
#   n = v + o = 4
#   N = k*n = 8
#   k*o = 4 >= m = 4 olduğu için imzalama lineer sistemi çözülebilir boyuttadır.
my_q = 7
my_v = 2
my_o = 2
my_m = 4
my_k = 2

mayo = MAYOSignatureSchemeDemo(q=my_q, v=my_v, o=my_o, m=my_m, k=my_k)
mayo.generate_keys()




MAYO İMZA ŞEMASI: TEZ SİMÜLASYONU 
Parametreler: GF(7), v=2, o=2, m=4
Whipping (k): 2 => Genişletilmiş Uzay Boyutu N = 8
Gerekli E Matrisi Sayısı (k(k+1)/2): 3
[ONAY] k(k+1)/2 < m (3 < 4). Şart sağlanıyor.
[ONAY] k*o >= m (4 >= 4). İmzalama için yeterli bilinmeyen sayısı mevcut.

[SİSTEM] Cisim Genişlemesi F_7^4 oluşturuldu.
         İndirgenemez Polinom: x^4 + 5*x^2 + 4*x + 3

**************************************** BÖLÜM 1: ANAHTAR ÜRETİMİ ****************************************

[1.1] Whipping (E) Matrislerinin Hesaplanması
      Yöntem: F_q[z]/f(z) üzerindeki çarpım matrisinin kuvvetleri
      Temel E Matrisi (alpha ile çarpımın F_q-lineer matris temsili):
[0 0 0 4]
[1 0 0 3]
[0 1 0 2]
[0 0 1 0]

      -> E_1,1 = E^0:
[1 0 0 0]
[0 1 0 0]
[0 0 1 0]
[0 0 0 1]

      -> E_1,2 = E^1:
[0 0 0 4]
[1 0 0 3]
[0 1 0 2]
[0 0 1 0]

      -> E_2,2 = E^2:
[0 0 4 0]
[0 0 3 4]
[1 0 2 3]
[0 1 0 2]
      Toplam 3 adet E matrisi hafızaya alındı.

[1.2] Gizli Oil Uzayı (O Matrisi)
      Boyut: 2x2


In [4]:
# =========================================================================================================================================
# İMZALAMA SENARYOSU (Bu hücrede bir önceki hücrede üretilen anahtar üzerinden imzalama rastgele bir mesaj için imzalama süreci ifade edilmektedir.
# =================================================================================================================================

msg = random_vector(mayo.F, my_m)
sig = mayo.sign(msg)

if sig is not None:
    mayo.verify(msg, sig)
else:
    print("İmza üretilemedi (Parametreler çok sınırda olabilir).")



**************************************** BÖLÜM 2: İMZALAMA SÜRECİ ****************************************
   İmzalanacak Özet Değer (t): (4, 2, 6, 3)

   --- Deneme 1 ---
   [2.1] Rastgele Vinegar Vektörleri (v_1...v_2):
         v_1: (4, 3)
         v_2: (1, 6)
   [2.2] Mevcut P*(v) Değeri y: (4, 5, 1, 2)
         Hedef Fark (RHS = t - y): (0, 4, 5, 1)
   [2.3] Lineer Sistem Matrisi L (Boyut: 4 x 4) Kuruluyor...
         Oluşturulan L Matrisi:
[2 3 3 0]
[5 4 6 3]
[0 6 3 4]
[6 0 0 6]
         Açık Lineer Denklem Sistemi:
            Denklem 1: (2)*x1_0 + (3)*x1_1 + (3)*x2_0 = 0
            Denklem 2: (5)*x1_0 + (4)*x1_1 + (6)*x2_0 + (3)*x2_1 = 4
            Denklem 3: (6)*x1_1 + (3)*x2_0 + (4)*x2_1 = 5
            Denklem 4: (6)*x1_0 + (6)*x2_1 = 1
   [2.4] ÇÖZÜM BAŞARILI! L * x = RHS sağlandı.
         Bulunan x (Oil Değerleri): (6, 1, 2, 0)
   [2.5] İmza Oluşturuluyor: s_i = (v_i + O*x_i) || x_i
         Blok 1:
           x_1: (6, 1)
           O*x_1: (6, 2)
           Üst Kısım (